<a href="https://colab.research.google.com/github/DariiaIvanova23/ml-lab-1/blob/main/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/DariiaIvanova23/ml-lab-1.git
%cd ml-lab-1

Cloning into 'ml-lab-1'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 91.69 KiB | 4.83 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/ml-lab-1


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

np.random.seed(42)
df = pd.read_csv('data/train.csv')

# Логарифмуємо цільову змінну
df['SalePrice'] = np.log1p(df['SalePrice'])

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

## Розбиття вибірки та запобігання Data Leakage

**Питання: Чому ми повинні розбити дані до, а не після масштабування чи кодування? Що таке Data Leakage?**

Data Leakage (Витік даних) — це потрапляння інформації з валідаційного або тестового набору в тренувальний процес.
Якщо порахувати середнє значення ($\mu$) для стандартизації чи знайти унікальні категорії на всьому датасеті разом, модель отримає непрямі підказки про тестовий розподіл. Це призведе до надмірно оптимістичних метрик під час розробки, але значного погіршення якості на нових реальних даних.
Розбиття на тренувальні, валідаційні та тестові дані проводиться на самому початку і усі трансформації обчислюють свої параметри виключно на тренувальних.

In [ ]:
# Розбиття на TRAIN, VAL, TEST
def train_val_test_split(df, train_ratio=0.70, val_ratio=0.15, random_seed=42):
    np.random.seed(random_seed)
    shuffled_indices = np.random.permutation(len(df))

    train_end = int(train_ratio * len(df))
    val_end = int((train_ratio + val_ratio) * len(df))

    train_idx = shuffled_indices[:train_end]
    val_idx = shuffled_indices[train_end:val_end]
    test_idx = shuffled_indices[val_end:]

    return df.iloc[train_idx].copy(), df.iloc[val_idx].copy(), df.iloc[test_idx].copy()

train_df, val_df, test_df = train_val_test_split(df)
print(f"Розміри вибірок: Train={train_df.shape[0]}, Val={val_df.shape[0]}, Test={test_df.shape[0]}")

## Відбір ознак та кодування категорій

Ordinal Encoding: Використовуємо для ознаки KitchenQual, оскільки вона має впорядкування якості (Po < Fa < TA < Gd < Ex -> 1, 2, 3, 4, 5).

One-Hot Encoding (OHE): Використовуємо для номінальної ознаки BldgType з параметром drop_first=True, щоб уникнути лінійної залежності стовпчиків (пастки фіктивних змінних / dummy variable trap).

Обробка невідомих категорій у Validation/Test: У числових ознаках пропуски заповнюються медіаною з Train. У Ordinal ознаках невідомі значення отримують базовий рівень 3 (середня якість). У OHE невідомі категорії просто кодуються нулями у всіх створених бінарних колонках.

In [ ]:
# ВІДБІР ТА ОБРОБКА ОЗНАК
num_cols = ['GrLivArea', 'YearBuilt', 'TotalBsmtSF', 'LotArea']
ord_cols = {'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}}
nom_cols = ['BldgType']
target_col = 'SalePrice'

# Заповнення пропусків числових ознак (медіаною виключно з Train)
for col in num_cols:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    val_df[col] = val_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)

# Ordinal Encoding для KitchenQual
for col, mapping in ord_cols.items():
    median_val = 3 # значення для невідомих/пропущених категорій
    train_df[col] = train_df[col].map(mapping).fillna(median_val)
    val_df[col] = val_df[col].map(mapping).fillna(median_val)
    test_df[col] = test_df[col].map(mapping).fillna(median_val)

# One-Hot Encoding для BldgType (категорії фіксуємо ТІЛЬКИ з Train)
ohe_categories = {col: sorted(train_df[col].dropna().unique()) for col in nom_cols}

def apply_ohe(df_in, cat_dict):
    df_out = df_in.copy()
    for col, categories in cat_dict.items():
        # drop_first=True для уникнення мультиколінеарності (dummy variable trap)
        for cat in categories[1:]:
            df_out[f"{col}_{cat}"] = (df_out[col] == cat).astype(float)
        df_out.drop(columns=[col], inplace=True)
    return df_out

train_df = apply_ohe(train_df, ohe_categories)
val_df = apply_ohe(val_df, ohe_categories)
test_df = apply_ohe(test_df, ohe_categories)

features = [c for c in train_df.columns if c in num_cols + list(ord_cols.keys()) or c.startswith('BldgType_')]

X_train_raw = train_df[features].values.astype(float)
y_train = train_df[target_col].values.astype(float)

X_val_raw = val_df[features].values.astype(float)
y_val = val_df[target_col].values.astype(float)

X_test_raw = test_df[features].values.astype(float)
y_test = test_df[target_col].values.astype(float)

## Масштабування числових ознак (Standardization)

Застосовуємо Z-score стандартизацію:
$$x' = \frac{x - \mu}{\sigma}$$
Параметри $\mu$ (середнє) та $\sigma$ (стандартне відхилення) обчислюються лише на тренувальний вибірці, після чого використовуються для масштабування валідаційної та тестової.

Геометричний зміст: Без масштабування лінії рівня функції втрат витягнуті у формі вузьких еліптичних впадин. Градієнт коливається між схилами, змушуючи обирати мізерний крок навчання. Стандартизація робить лінії рівня концентричними колами, що забезпечує швидкий і прямий спуск до мінімуму.

In [ ]:
# СТАНДАРТИЗАЦІЯ
class CustomStandardScaler:
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        self.std_[self.std_ == 0.0] = 1.0 # захист від ділення на 0
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

scaler = CustomStandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

## Реалізація лінійної регресії

Модель: $\hat{y} = Xw + b$

Функція втрат (MSE):
  $$J(w, b) = \frac{1}{2m} (\hat{y} - y)^T (\hat{y} - y)$$

Векторні градієнти:
  $$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y), \quad \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$
*текст курсивом*

In [ ]:
# ЛІНІЙНА РЕГРЕСІЯ
class LinearRegression:
    def __init__(self):
        self.w = None
        self.b = None
        self.loss_history = []

    def predict(self, X):
        return np.dot(X, self.w) + self.b

    def compute_cost(self, y_pred, y_true):
        m = len(y_true)
        error = y_pred - y_true
        return (1.0 / (2.0 * m)) * np.dot(error, error)

    def compute_gradients(self, X, y_pred, y_true):
        m = len(y_true)
        error = y_pred - y_true
        dw = (1.0 / m) * np.dot(X.T, error)
        db = (1.0 / m) * np.sum(error)
        return dw, db

    def fit(self, X, y, method='batch', lr=0.01, epochs=100, batch_size=32):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0
        self.loss_history = []

        for epoch in range(epochs):
            if method == 'batch':
                y_pred = self.predict(X)
                dw, db = self.compute_gradients(X, y_pred, y)
                self.w -= lr * dw
                self.b -= lr * db

            elif method == 'sgd':
                indices = np.random.permutation(m)
                for idx in indices:
                    xi = X[idx:idx+1]
                    yi = y[idx:idx+1]
                    y_pred_i = self.predict(xi)
                    dw, db = self.compute_gradients(xi, y_pred_i, yi)
                    self.w -= lr * dw
                    self.b -= lr * db

            elif method == 'mini_batch':
                indices = np.random.permutation(m)
                X_shuffled = X[indices]
                y_shuffled = y[indices]

                for start_idx in range(0, m, batch_size):
                    end_idx = min(start_idx + batch_size, m)
                    X_batch = X_shuffled[start_idx:end_idx]
                    y_batch = y_shuffled[start_idx:end_idx]

                    y_pred_batch = self.predict(X_batch)
                    dw, db = self.compute_gradients(X_batch, y_pred_batch, y_batch)
                    self.w -= lr * dw
                    self.b -= lr * db

            current_loss = self.compute_cost(self.predict(X), y)
            self.loss_history.append(current_loss)

        return self

## Градієнтний спуск та порівняння трьох варіантів оптимізації

Batch GD: оновлення один раз за епоху за всією вибіркою.

SGD: оновлення ваг після кожного окремого прикладу.

Mini-batch GD (B=32): оновлення після пакету з 32 прикладів.

In [ ]:
# НАВЧАННЯ ТА ПОРІВНЯННЯ МЕТОДІВ ОПТИМІЗАЦІЇ
epochs = 150
lr = 0.02

print("\nНавчання варіантів Градієнтного Спуску")
model_bgd = LinearRegression().fit(X_train, y_train, method='batch', lr=lr, epochs=epochs)
model_sgd = LinearRegression().fit(X_train, y_train, method='sgd', lr=0.001, epochs=epochs)
model_mbgd = LinearRegression().fit(X_train, y_train, method='mini_batch', lr=lr, epochs=epochs, batch_size=32)

plt.figure(figsize=(10, 6))
plt.plot(model_bgd.loss_history, label='Batch GD (lr=0.02)', color='blue', linewidth=2)
plt.plot(model_mbgd.loss_history, label='Mini-batch GD (B=32, lr=0.02)', color='green', linewidth=1.5)
plt.plot(model_sgd.loss_history, label='SGD (lr=0.001)', color='orange', alpha=0.8)
plt.title('Порівняння збіжності функцій втрат (Cost vs Epochs)')
plt.xlabel('Епоха')
plt.ylabel('Cost (MSE / 2)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Висновки щодо оптимізації:
Batch GD найкраще збігається та демонструє ідеально монотонну, гладку траєкторію без коливань. SGD має найбільшу стохастичність і флуктуації біля оптимуму. Mini-batch GD (B=32) забезпечує оптимальний баланс — збігається значно швидше за Batch GD за кількістю епох і працює стабільніше за SGD.SGD вимагає меншого $\alpha=0.001$ для стабільності, тоді як Batch GD при $\alpha=0.02$ потребує значно більше 150 кроків для досягнення мінімуму.


### Оцінка якості та коефіцієнт детермінації ($R^2$)

$$R^2 = 1 - \frac{\sum (y^{(i)} - \hat{y}^{(i)})^2}{\sum (y^{(i)} - \bar{y})^2} = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$$

$R^2$ показує частку дисперсії цільової змінної, пояснену моделлю.
Якщо модель прогнозує гірше за просте середнє арифметичне ($SS_{\text{res}} > SS_{\text{tot}}$), то це значення може бути від'ємним, що свідчить про недонавчання або розбіжність.

In [ ]:
# ОБЧИСЛЕННЯ R^2
def r2_score_numpy(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot)

def evaluate_model(model, name):
    r2_train = r2_score_numpy(y_train, model.predict(X_train))
    r2_val   = r2_score_numpy(y_val, model.predict(X_val))
    r2_test  = r2_score_numpy(y_test, model.predict(X_test))
    print(f"[{name}] R^2 Train: {r2_train:.4f} | Val: {r2_val:.4f} | Test: {r2_test:.4f}")

print("\nРезультати метрики R^2")
evaluate_model(model_bgd, "Batch GD")
evaluate_model(model_mbgd, "Mini-batch GD")
evaluate_model(model_sgd, "SGD")

### Висновки щодо оцінки якості ($R^2$):
Mini-batch GD та SGD показали високі результати: $R^2 \approx 0.82 - 0.83$ на тесті. Порівняння $R^2$ на тренувальних та тестових даних свідчить про відсутність перенавчання.

В Batch GD $R^2 < 0$ ($-1.86$) тому що за 150 епох Batch GD зробив лише 150 оновлень ваг, тому не встиг дійти до мінімуму і зупинився в стані недонавчання.

---

## Перевірка статистичних припущень
Аналізуємо залишки $\epsilon = y - \hat{y}$ моделі Mini-batch GD:
1. **Лінійність:** Графік Residuals vs Fitted.
2. **Нормальність залишків:** Q-Q Plot.
3. **Гомоскедастичність:** Графік Scale-Location.
4. **Мультиколінеарність:** Variance Inflation Factor ($VIF_j = (R^{-1})_{jj}$).

In [ ]:
# ПЕРЕВІРКА СТАТИСТИЧНИХ ПРИПУЩЕНЬ
best_model = model_mbgd
y_test_pred = best_model.predict(X_test)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Linearity
axes[0].scatter(y_test_pred, residuals, alpha=0.6, color='purple')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('1. Linearity: Residuals vs Fitted')
axes[0].set_xlabel('Прогнозовані значення (log scale)')
axes[0].set_ylabel('Залишки (Residuals)')
axes[0].grid(True, alpha=0.3)

# 2. Normality
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('2. Normality: Q-Q Plot')
axes[1].grid(True, alpha=0.3)

# 3. Homoscedasticity
std_residuals = residuals / np.std(residuals)
axes[2].scatter(y_test_pred, np.sqrt(np.abs(std_residuals)), alpha=0.6, color='teal')
axes[2].set_title('3. Homoscedasticity: Scale-Location')
axes[2].set_xlabel('Прогнозовані значення')
axes[2].set_ylabel('sqrt(|Standardized Residuals|)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Multicollinearity (VIF)
corr_matrix = np.corrcoef(X_train, rowvar=False)
try:
    inv_corr = np.linalg.inv(corr_matrix)
    vif_values = np.diag(inv_corr)
    print("\n--- Variance Inflation Factor (VIF) ---")
    for feat, vif in zip(features, vif_values):
        print(f"Ознака '{feat}': VIF = {vif:.2f}")
except np.linalg.LinAlgError:
    print("Кореляційна матриця сингулярна.")

Припущення про лінійність виконується. Залишки рівномірно і хаотично зосереджені навколо горизонтальної нульової лінії без параболічних чи інших системних відхилень. Це означає, що специфікація лінійної моделі є адекватною.

Припущення про нормальність залишків також виконується, тому що на графіку Q-Q точки лягають уздовж теоретичної прямої під кутом 45° (відхилення є лише на крайніх квантилях, що типово для ринкових викидів). Це означає, що оцінки коефіцієнтів та довірчі інтервали є надійними.

Припущення про гомоскедастичність виконується, тому що на графіку Scale-Location дисперсія залишків не утворює форму конуса при збільшенні ціни. Це означає, що похибка моделі однакова для різних цінових діапазонів житла.

Припущення про відсутність мультиколінеарності виконується, тому що всі показники VIF знаходяться в межах 1.03 – 1.74, що набагато менше рівня 5.0. Це означає, що ваги моделі є стійкими, не мають сильної дисперсії та легко інтерпретуються.

---

## Експеримент В: Ridge Regression (L2-регуляризація)

Додаємо регуляризаційний штраф до функції втрат:
$$J_{\text{ridge}}(w, b) = \frac{1}{2m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})^2 + \frac{\lambda}{2m} \|w\|^2$$
Виведений градієнт для ваг $w$:
$$\frac{\partial J_{\text{ridge}}}{\partial w} = \frac{1}{m} X^T (\hat{y} - y) + \frac{\lambda}{m} w$$

In [ ]:
# RIDGE REGRESSION
class RidgeRegression(LinearRegression):
    def __init__(self, l2_lambda=10.0):
        super().__init__()
        self.l2_lambda = l2_lambda

    def compute_cost(self, y_pred, y_true):
        m = len(y_true)
        mse_cost = super().compute_cost(y_pred, y_true)
        l2_penalty = (self.l2_lambda / (2.0 * m)) * np.dot(self.w, self.w)
        return mse_cost + l2_penalty

    def compute_gradients(self, X, y_pred, y_true):
        m = len(y_true)
        dw, db = super().compute_gradients(X, y_pred, y_true)
        dw_regularized = dw + (self.l2_lambda / m) * self.w
        return dw_regularized, db

print("\nL2 Regularization (Ridge)")
ridge_model = RidgeRegression(l2_lambda=20.0).fit(
    X_train, y_train, method='mini_batch', lr=lr, epochs=epochs, batch_size=32
)

plt.figure(figsize=(10, 5))
x_axis = np.arange(len(features))
width = 0.35
plt.bar(x_axis - width/2, model_mbgd.w, width=width, label='Linear (Без регуляризації)', color='royalblue')
plt.bar(x_axis + width/2, ridge_model.w, width=width, label='Ridge (L2 lambda=20)', color='crimson')
plt.xticks(x_axis, features, rotation=45, ha='right')
plt.ylabel('Значення ваг (w)')
plt.title('Ефект L2-регуляризації (Shrinkage)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Висновки за Експериментом:
Графік показує ефект стягування ваг (Shrinkage): під дією $\lambda = 20$ коефіцієнти для ознак GrLivArea, YearBuilt та KitchenQual зменшилися за абсолютною величиною. Регуляризація запобігає домінуванню окремих ознак і знижує дисперсію прогнозу моделі.